# Tenor & Rate Distribution Analytics

## Comprehensive Analysis of Swap Tenors and Traded Rates

This notebook provides detailed analysis of:

1. **Tenor Distribution** - Detailed breakdown by maturity
2. **Rate Distribution** - Fixed rate analysis by tenor
3. **Rate Evolution** - Intraday rate movements
4. **Spread Analysis** - Spread over curve benchmarks
5. **Forward Start Analysis** - Forward starting swap activity
6. **Benchmark Tenors** - Focus on key tenors (2Y, 5Y, 10Y, 30Y)

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
from scipy import stats

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York")
CHI_tz = pytz.timezone("America/Chicago")
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.SDRDataBuilder import SDRDataBuilder

cache_path = r"/tmp/sdr_cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)

## 1. Data Loading

In [ ]:
start = NY_tz.localize(datetime.datetime(2025, 12, 19, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 19, 17, 0))

raw_df = sdr.grab_sdr_trades(
    start_timestamp=start,
    end_timestamp=end,
    agency="CFTC",
    asset_class="RATES"
)

print(f"Total trades fetched: {len(raw_df):,}")

In [ ]:
def preprocess_tenor_rate_data(df: pd.DataFrame) -> pd.DataFrame:
    """Preprocess data for tenor and rate analysis."""
    df = df.copy()
    
    # Filter to new trades only
    df = df[df['Action type'] == 'NEWT'].copy()
    
    # Filter USD
    df = df[df['Notional currency-Leg 1'] == 'USD'].copy()
    
    # Filter SOFR swaps
    sofr_mask = df['UPI Underlier Name'].str.contains('SOFR', case=False, na=False)
    df = df[sofr_mask].copy()
    
    # Filter to swaps (not options)
    swap_mask = df['UPI FISN'].str.contains('Swap|OIS', case=False, na=False)
    option_mask = df['UPI FISN'].str.contains('Call|Put|Cap|Floor|Swaption', case=False, na=False)
    df = df[swap_mask & ~option_mask].copy()
    
    # Parse dates
    df['Event timestamp'] = pd.to_datetime(df['Event timestamp'], utc=True)
    df['Execution Timestamp'] = pd.to_datetime(df['Execution Timestamp'], utc=True)
    df['Effective Date'] = pd.to_datetime(df['Effective Date'], errors='coerce')
    df['Expiration Date'] = pd.to_datetime(df['Expiration Date'], errors='coerce')
    df['Event_Time_NY'] = df['Event timestamp'].dt.tz_convert('America/New_York')
    
    # Calculate tenor
    df['Tenor_Days'] = (df['Expiration Date'] - df['Effective Date']).dt.days
    df['Tenor_Years'] = df['Tenor_Days'] / 365.25
    df['Tenor_Months'] = df['Tenor_Days'] / 30.44
    
    # Calculate days to effective (forward start analysis)
    today = datetime.date.today()
    df['Days_To_Effective'] = (df['Effective Date'].dt.date - today).apply(lambda x: x.days if pd.notna(x) else np.nan)
    df['Is_Forward_Start'] = df['Days_To_Effective'] > 5  # More than T+5
    
    # Parse fixed rate
    df['Fixed rate-Leg 1'] = pd.to_numeric(df['Fixed rate-Leg 1'], errors='coerce')
    df['Fixed_Rate_Pct'] = df['Fixed rate-Leg 1'] * 100
    
    # Parse notional
    df['Notional amount-Leg 1'] = df['Notional amount-Leg 1'].astype(str).str.replace(',', '')
    df['Notional amount-Leg 1'] = pd.to_numeric(df['Notional amount-Leg 1'], errors='coerce')
    
    # Create detailed tenor buckets
    def detailed_tenor_bucket(years):
        if pd.isna(years) or years <= 0:
            return 'Unknown'
        elif years <= 0.25:
            return '0-3M'
        elif years <= 0.5:
            return '3-6M'
        elif years <= 0.75:
            return '6-9M'
        elif years <= 1:
            return '9M-1Y'
        elif years <= 1.5:
            return '1-1.5Y'
        elif years <= 2:
            return '1.5-2Y'
        elif years <= 3:
            return '2-3Y'
        elif years <= 4:
            return '3-4Y'
        elif years <= 5:
            return '4-5Y'
        elif years <= 7:
            return '5-7Y'
        elif years <= 10:
            return '7-10Y'
        elif years <= 15:
            return '10-15Y'
        elif years <= 20:
            return '15-20Y'
        elif years <= 30:
            return '20-30Y'
        else:
            return '30Y+'
    
    df['Tenor_Bucket'] = df['Tenor_Years'].apply(detailed_tenor_bucket)
    
    # Identify benchmark tenors
    def is_benchmark_tenor(years):
        if pd.isna(years):
            return False
        benchmark_tenors = [1, 2, 3, 5, 7, 10, 15, 20, 30]
        return any(abs(years - t) < 0.1 for t in benchmark_tenors)
    
    def get_benchmark_tenor(years):
        if pd.isna(years):
            return None
        benchmark_tenors = [1, 2, 3, 5, 7, 10, 15, 20, 30]
        for t in benchmark_tenors:
            if abs(years - t) < 0.1:
                return f"{t}Y"
        return None
    
    df['Is_Benchmark'] = df['Tenor_Years'].apply(is_benchmark_tenor)
    df['Benchmark_Tenor'] = df['Tenor_Years'].apply(get_benchmark_tenor)
    
    return df.reset_index(drop=True)

df = preprocess_tenor_rate_data(raw_df)
print(f"Processed USD SOFR swaps: {len(df):,}")

## 2. Detailed Tenor Distribution

In [ ]:
# Tenor bucket order
tenor_order = ['0-3M', '3-6M', '6-9M', '9M-1Y', '1-1.5Y', '1.5-2Y', '2-3Y', '3-4Y', '4-5Y', 
               '5-7Y', '7-10Y', '10-15Y', '15-20Y', '20-30Y', '30Y+', 'Unknown']

# Distribution by tenor bucket
tenor_dist = df.groupby('Tenor_Bucket').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum',
    'Fixed_Rate_Pct': ['mean', 'median', 'std']
}).round(4)
tenor_dist.columns = ['Trade_Count', 'Total_Notional', 'Avg_Rate', 'Median_Rate', 'Rate_StdDev']
tenor_dist = tenor_dist.reindex([t for t in tenor_order if t in tenor_dist.index])

tenor_dist

In [ ]:
# Visualization: Trade count and notional by tenor
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Trade Count by Tenor', 'Total Notional ($B) by Tenor'])

fig.add_trace(
    go.Bar(x=tenor_dist.index, y=tenor_dist['Trade_Count'],
           name='Trade Count', marker_color='steelblue'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=tenor_dist.index, y=tenor_dist['Total_Notional']/1e9,
           name='Notional ($B)', marker_color='darkgreen'),
    row=2, col=1
)

fig.update_layout(height=700, title_text='Tenor Distribution Analysis', showlegend=False)
fig.show()

In [ ]:
# Scatter plot of actual tenor distribution
valid_tenors = df[df['Tenor_Years'].notna() & (df['Tenor_Years'] > 0) & (df['Tenor_Years'] <= 50)]

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=valid_tenors['Tenor_Years'],
    nbinsx=100,
    marker_color='steelblue',
    opacity=0.75
))

# Add vertical lines for benchmark tenors
benchmark_tenors = [1, 2, 3, 5, 7, 10, 15, 20, 30]
for t in benchmark_tenors:
    fig.add_vline(x=t, line_dash='dash', line_color='red', opacity=0.5,
                  annotation_text=f'{t}Y', annotation_position='top')

fig.update_layout(
    title='Tenor Distribution (in Years)',
    xaxis_title='Tenor (Years)',
    yaxis_title='Trade Count',
    height=500
)
fig.show()

## 3. Fixed Rate Distribution

In [ ]:
# Overall rate distribution
rates = df['Fixed_Rate_Pct'].dropna()
rates = rates[(rates > 0) & (rates < 15)]  # Filter outliers

print("Fixed Rate Statistics (%):")
print("="*50)
print(f"Count: {len(rates):,}")
print(f"Mean: {rates.mean():.4f}%")
print(f"Median: {rates.median():.4f}%")
print(f"Std Dev: {rates.std():.4f}%")
print(f"Min: {rates.min():.4f}%")
print(f"Max: {rates.max():.4f}%")

In [ ]:
# Rate distribution histogram
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=rates,
    nbinsx=100,
    marker_color='steelblue',
    opacity=0.75
))

fig.add_vline(x=rates.median(), line_dash='dash', line_color='red', line_width=2,
              annotation_text=f'Median: {rates.median():.3f}%')

fig.update_layout(
    title='Fixed Rate Distribution',
    xaxis_title='Fixed Rate (%)',
    yaxis_title='Trade Count',
    height=400
)
fig.show()

In [ ]:
# Rate by tenor - box plot
valid_df = df[(df['Fixed_Rate_Pct'] > 0) & (df['Fixed_Rate_Pct'] < 15) & df['Tenor_Bucket'].notna()].copy()

fig = go.Figure()

for bucket in [t for t in tenor_order if t in valid_df['Tenor_Bucket'].unique()]:
    bucket_data = valid_df[valid_df['Tenor_Bucket'] == bucket]['Fixed_Rate_Pct']
    if len(bucket_data) > 0:
        fig.add_trace(go.Box(
            y=bucket_data,
            name=bucket,
            boxpoints='outliers'
        ))

fig.update_layout(
    title='Fixed Rate Distribution by Tenor Bucket',
    yaxis_title='Fixed Rate (%)',
    height=600,
    showlegend=False
)
fig.show()

In [ ]:
# Scatter plot: Rate vs Tenor
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=valid_df['Tenor_Years'],
    y=valid_df['Fixed_Rate_Pct'],
    mode='markers',
    marker=dict(
        size=6,
        color='steelblue',
        opacity=0.5
    ),
    hovertemplate='Tenor: %{x:.2f}Y<br>Rate: %{y:.4f}%<extra></extra>'
))

# Add trend line
valid_tenor_rate = valid_df[['Tenor_Years', 'Fixed_Rate_Pct']].dropna()
if len(valid_tenor_rate) > 10:
    z = np.polyfit(valid_tenor_rate['Tenor_Years'], valid_tenor_rate['Fixed_Rate_Pct'], 3)
    p = np.poly1d(z)
    x_trend = np.linspace(valid_tenor_rate['Tenor_Years'].min(), valid_tenor_rate['Tenor_Years'].max(), 100)
    fig.add_trace(go.Scatter(
        x=x_trend,
        y=p(x_trend),
        mode='lines',
        name='Trend',
        line=dict(color='red', width=2)
    ))

fig.update_layout(
    title='Fixed Rate vs Tenor (Implied Swap Curve)',
    xaxis_title='Tenor (Years)',
    yaxis_title='Fixed Rate (%)',
    height=500
)
fig.show()

## 4. Benchmark Tenor Analysis

In [ ]:
# Focus on benchmark tenors
benchmark_df = df[df['Is_Benchmark']].copy()

if len(benchmark_df) > 0:
    benchmark_stats = benchmark_df.groupby('Benchmark_Tenor').agg({
        'Dissemination Identifier': 'count',
        'Notional amount-Leg 1': ['sum', 'mean'],
        'Fixed_Rate_Pct': ['mean', 'median', 'std', 'min', 'max']
    }).round(4)
    benchmark_stats.columns = ['Trade_Count', 'Total_Notional', 'Avg_Notional', 
                                'Avg_Rate', 'Median_Rate', 'Rate_StdDev', 'Min_Rate', 'Max_Rate']
    
    # Reorder by tenor
    tenor_order_benchmark = ['1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '15Y', '20Y', '30Y']
    benchmark_stats = benchmark_stats.reindex([t for t in tenor_order_benchmark if t in benchmark_stats.index])
    
    print("Benchmark Tenor Statistics:")
    display(benchmark_stats)
else:
    print("No benchmark tenor trades found")

In [ ]:
# Benchmark tenor rate ranges
if len(benchmark_df) > 0:
    fig = go.Figure()
    
    for tenor in ['2Y', '5Y', '10Y', '30Y']:
        tenor_data = benchmark_df[benchmark_df['Benchmark_Tenor'] == tenor]['Fixed_Rate_Pct'].dropna()
        if len(tenor_data) > 0:
            fig.add_trace(go.Violin(
                y=tenor_data,
                name=tenor,
                box_visible=True,
                meanline_visible=True
            ))
    
    fig.update_layout(
        title='Rate Distribution for Key Benchmark Tenors',
        yaxis_title='Fixed Rate (%)',
        height=500
    )
    fig.show()

## 5. Rate Evolution Analysis (Intraday)

In [ ]:
# Track rate evolution for benchmark tenors throughout the day
if len(benchmark_df) > 0:
    benchmark_df_sorted = benchmark_df.sort_values('Event_Time_NY')
    
    fig = go.Figure()
    
    for tenor in ['2Y', '5Y', '10Y', '30Y']:
        tenor_data = benchmark_df_sorted[benchmark_df_sorted['Benchmark_Tenor'] == tenor].copy()
        if len(tenor_data) > 0:
            fig.add_trace(go.Scatter(
                x=tenor_data['Event_Time_NY'],
                y=tenor_data['Fixed_Rate_Pct'],
                mode='markers',
                name=tenor,
                marker=dict(size=8)
            ))
    
    fig.update_layout(
        title='Intraday Rate Evolution by Benchmark Tenor',
        xaxis_title='Time (NY)',
        yaxis_title='Fixed Rate (%)',
        height=500,
        legend=dict(x=0.02, y=0.98)
    )
    fig.show()

In [ ]:
# Calculate intraday rate ranges by benchmark tenor
if len(benchmark_df) > 0:
    intraday_range = benchmark_df.groupby('Benchmark_Tenor')['Fixed_Rate_Pct'].agg(
        ['min', 'max', lambda x: x.max() - x.min()]
    ).round(4)
    intraday_range.columns = ['Low', 'High', 'Range']
    intraday_range['Range_bps'] = intraday_range['Range'] * 100
    intraday_range = intraday_range.reindex([t for t in tenor_order_benchmark if t in intraday_range.index])
    
    print("Intraday Rate Ranges by Benchmark Tenor:")
    display(intraday_range)

## 6. Forward Start Analysis

In [ ]:
# Forward start trade analysis
forward_stats = df.groupby('Is_Forward_Start').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': ['sum', 'mean'],
    'Days_To_Effective': 'mean'
}).round(2)
forward_stats.columns = ['Trade_Count', 'Total_Notional', 'Avg_Notional', 'Avg_Days_To_Effective']
forward_stats.index = forward_stats.index.map({True: 'Forward Start', False: 'Spot Start'})

print("Forward Start Trade Analysis:")
display(forward_stats)

In [ ]:
# Distribution of forward start periods
forward_trades = df[df['Is_Forward_Start']].copy()

if len(forward_trades) > 0:
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=forward_trades['Days_To_Effective'],
        nbinsx=50,
        marker_color='steelblue'
    ))
    
    fig.update_layout(
        title='Forward Start Period Distribution',
        xaxis_title='Days to Effective Date',
        yaxis_title='Trade Count',
        height=400
    )
    fig.show()
else:
    print("No forward start trades in dataset")

In [ ]:
# IMM start dates analysis
# IMM dates are typically 3rd Wednesday of Mar, Jun, Sep, Dec
if len(forward_trades) > 0:
    forward_trades['Effective_Month'] = forward_trades['Effective Date'].dt.month
    forward_trades['Effective_Day'] = forward_trades['Effective Date'].dt.day
    
    # Count by month
    month_dist = forward_trades.groupby('Effective_Month')['Dissemination Identifier'].count()
    
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
        y=[month_dist.get(i, 0) for i in range(1, 13)],
        marker_color='steelblue'
    ))
    
    # Highlight IMM months
    for i, month in enumerate(['Mar', 'Jun', 'Sep', 'Dec']):
        fig.add_vrect(x0=i*3+1.5, x1=i*3+2.5, fillcolor='lightgreen', opacity=0.3, line_width=0)
    
    fig.update_layout(
        title='Forward Start Trades by Effective Month (IMM months highlighted)',
        xaxis_title='Effective Month',
        yaxis_title='Trade Count',
        height=400
    )
    fig.show()

## 7. Curve Spread Analysis

In [ ]:
# Calculate spreads between benchmark tenors
if len(benchmark_df) > 0:
    # Get average rates for each benchmark tenor
    avg_rates = benchmark_df.groupby('Benchmark_Tenor')['Fixed_Rate_Pct'].median()
    
    # Calculate common spreads
    spreads = {}
    
    if '2Y' in avg_rates.index and '10Y' in avg_rates.index:
        spreads['2s10s'] = (avg_rates['10Y'] - avg_rates['2Y']) * 100
    
    if '5Y' in avg_rates.index and '30Y' in avg_rates.index:
        spreads['5s30s'] = (avg_rates['30Y'] - avg_rates['5Y']) * 100
    
    if '2Y' in avg_rates.index and '5Y' in avg_rates.index:
        spreads['2s5s'] = (avg_rates['5Y'] - avg_rates['2Y']) * 100
    
    if '10Y' in avg_rates.index and '30Y' in avg_rates.index:
        spreads['10s30s'] = (avg_rates['30Y'] - avg_rates['10Y']) * 100
    
    print("Curve Spreads (bps):")
    print("="*40)
    for name, value in spreads.items():
        print(f"{name}: {value:.1f} bps")

In [ ]:
# Visualize the average curve
if len(benchmark_df) > 0:
    avg_rates_df = avg_rates.reset_index()
    avg_rates_df.columns = ['Tenor', 'Rate']
    avg_rates_df['Tenor_Numeric'] = avg_rates_df['Tenor'].str.replace('Y', '').astype(float)
    avg_rates_df = avg_rates_df.sort_values('Tenor_Numeric')
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=avg_rates_df['Tenor_Numeric'],
        y=avg_rates_df['Rate'],
        mode='lines+markers',
        name='Median Rate',
        line=dict(color='steelblue', width=2),
        marker=dict(size=10)
    ))
    
    fig.update_layout(
        title='SOFR Swap Curve (Median Traded Rates)',
        xaxis_title='Tenor (Years)',
        yaxis_title='Rate (%)',
        height=500
    )
    fig.update_xaxes(tickvals=avg_rates_df['Tenor_Numeric'], ticktext=avg_rates_df['Tenor'])
    fig.show()

## 8. Summary Report

In [ ]:
print("="*70)
print("TENOR & RATE DISTRIBUTION SUMMARY")
print("="*70)
print(f"\nAnalysis Period: {start.strftime('%Y-%m-%d %H:%M')} to {end.strftime('%Y-%m-%d %H:%M')} NY")
print(f"\nTotal USD SOFR Swaps: {len(df):,}")

print(f"\nTenor Distribution:")
top_tenors = df.groupby('Tenor_Bucket')['Dissemination Identifier'].count().sort_values(ascending=False).head(5)
for tenor, count in top_tenors.items():
    print(f"  {tenor}: {count:,} trades ({count/len(df)*100:.1f}%)")

if len(benchmark_df) > 0:
    print(f"\nBenchmark Tenor Trades: {len(benchmark_df):,} ({len(benchmark_df)/len(df)*100:.1f}%)")

print(f"\nRate Statistics:")
valid_rates = df['Fixed_Rate_Pct'].dropna()
valid_rates = valid_rates[(valid_rates > 0) & (valid_rates < 15)]
print(f"  Trades with valid rate: {len(valid_rates):,}")
print(f"  Rate Range: {valid_rates.min():.4f}% - {valid_rates.max():.4f}%")
print(f"  Median Rate: {valid_rates.median():.4f}%")

print(f"\nForward Start Trades: {len(df[df['Is_Forward_Start']]):,} ({len(df[df['Is_Forward_Start']])/len(df)*100:.1f}%)")